# Five-Company Global Graph Prototype

Read-only prototype on the real `./processed/transaction_parquet.parquet`: the 5 most-active companies plus their top insiders as one shared-node bipartite network, hub = highest-degree node, PyVis force-directed HTML saved to `./processed/five_company_global_graph.html`.
This notebook modifies nothing: no existing notebook, parquet file, or production pipeline is touched.

## Cell 1 — Schema inspection

In [ ]:
# %pip install -q networkx pyvis   (run this line manually in Jupyter if networkx/pyvis are missing)
import os
import polars as pl

TXN_PATH = "./processed/transaction_parquet.parquet"
COMPANY_META = "./processed/company_metadata.parquet"
INSIDER_META = "./processed/insider_metadata.parquet"
HTML_OUT = "./processed/five_company_global_graph.html"

# Lazy schema inspection — nothing is loaded into memory.
#   company/issuer      -> issuerCik
#   insider/owner       -> rptOwnerCik (relationship via shared accessionNumber + role flags)
#   dates               -> transactionDate / filingDate
#   amount/value        -> transactionShares * transactionPricePerShare
for label, path in [("transactions", TXN_PATH), ("company metadata", COMPANY_META), ("insider metadata", INSIDER_META)]:
    print(f"--- {label}: {path} ---")
    print(pl.scan_parquet(path).collect_schema())


## Cell 2 — Select exactly 5 companies

In [ ]:
import polars as pl

# Top-5 issuerCik by transaction count: one lazy aggregation, only 5 rows collected.
top5 = (
    pl.scan_parquet(TXN_PATH)
    .group_by("issuerCik")
    .agg(pl.len().alias("n_txn"))
    .sort("n_txn", descending=True)
    .head(5)
    .collect()
)
TARGET_CIKS = top5["issuerCik"].to_list()

# Real companyName/ticker from company_metadata.parquet (no fake data).
labels = (
    pl.scan_parquet(COMPANY_META)
    .filter(pl.col("issuerCik").is_in(TARGET_CIKS))
    .collect()
    .with_columns(
        pl.coalesce(["ticker", pl.col("issuerCik").cast(pl.String)]).alias("ticker"),
        pl.coalesce(["companyName", pl.col("issuerCik").cast(pl.String)]).alias("companyName"),
    )
    .to_dicts()
)
LABEL = {r["issuerCik"]: r for r in labels}

print("5 selected companies (issuerCik | ticker | companyName | n_txn):")
for cik in TARGET_CIKS:
    n = top5.filter(pl.col("issuerCik") == cik)["n_txn"][0]
    print(f"  {cik} | {LABEL[cik]['ticker']} | {LABEL[cik]['companyName']} | {n:,} txns")


## Cell 3 — Construct the graph

In [ ]:
import networkx as nx
import polars as pl

# Top-8 insiders per company by total trade value (shares x price).
# A shared rptOwnerCik keeps each of its company edges but exists as ONE node.
pairs = (
    pl.scan_parquet(TXN_PATH)
    .filter(pl.col("issuerCik").is_in(TARGET_CIKS))
    .with_columns(
        (pl.col("transactionShares") * pl.col("transactionPricePerShare"))
        .fill_null(0.0).alias("value")
    )
    .group_by(["issuerCik", "rptOwnerCik"])
    .agg([pl.len().alias("n_txn"), pl.col("value").sum().alias("total_value")])
    .collect()
    .sort(["issuerCik", "total_value"], descending=[False, True])
    .group_by("issuerCik", maintain_order=True)
    .head(8)
)
print(f"selected insider-company pairs: {pairs.height}")

# Real insider names from insider_metadata.parquet (no fake data).
names = (
    pl.scan_parquet(INSIDER_META)
    .filter(pl.col("rptOwnerCik").is_in(pairs["rptOwnerCik"].unique().to_list()))
    .select(["rptOwnerCik", "rptOwnerName"])
    .collect()
)
INAME = dict(zip(names["rptOwnerCik"].to_list(), names["rptOwnerName"].to_list()))

G = nx.Graph()
for cik in TARGET_CIKS:
    G.add_node(f"C:{cik}", kind="company", cik=cik,
               label=f"{LABEL[cik]['ticker']}\n{LABEL[cik]['companyName']}",
               title=f"{LABEL[cik]['companyName']} ({LABEL[cik]['ticker']}, CIK {cik})")
for row in pairs.to_dicts():
    ikey = f"I:{row['rptOwnerCik']}"
    if not G.has_node(ikey):  # shared insider -> single node, edges to each company kept
        nm = INAME.get(row["rptOwnerCik"]) or str(row["rptOwnerCik"])
        G.add_node(ikey, kind="insider", cik=row["rptOwnerCik"], label=nm,
                   title=f"{nm} (CIK {row['rptOwnerCik']})")
    G.add_edge(f"C:{row['issuerCik']}", ikey,
               n_txn=row["n_txn"], total_value=row["total_value"])

n_co = sum(1 for _, d in G.nodes(data=True) if d["kind"] == "company")
print(f"nodes: {G.number_of_nodes()} ({n_co} companies, {G.number_of_nodes() - n_co} insiders) | edges: {G.number_of_edges()}")


## Cell 4 — Identify the hub

In [ ]:
# Degree from the actual constructed graph — the hub may be a company OR a shared insider.
deg = dict(G.degree())
hub, hub_deg = max(deg.items(), key=lambda kv: kv[1])
print("degree per company:")
for cik in TARGET_CIKS:
    print(f"  {LABEL[cik]['ticker']} (CIK {cik}): degree {deg[f'C:{cik}']}")
print(f"HUB: {hub} | label={G.nodes[hub]['label']!r} | kind={G.nodes[hub]['kind']} | degree={hub_deg}")
print(f"TOTAL nodes={G.number_of_nodes()} edges={G.number_of_edges()}")


## Cell 5 — Visualization

In [ ]:
from pyvis.network import Network
import math

# Force-directed Barnes-Hut layout. Companies big green, insiders small blue,
# hub gold + enlarged. Edge width = log-scaled total trade value.
vs = [d["total_value"] for _, _, d in G.edges(data=True)]
lo, hi = min(vs), max(vs)
span = (math.log1p(hi) - math.log1p(lo)) or 1.0

net = Network(height="750px", width="100%", bgcolor="#111111", font_color="white")
net.barnes_hut(gravity=-8000, central_gravity=0.3, spring_length=250, spring_strength=0.01, damping=0.09)
for n, d in G.nodes(data=True):
    if n == hub:
        net.add_node(n, label="HUB\n" + d["label"], title=f"HUB {d['title']} | degree {deg[n]}",
                     color="#FFC107", size=50, borderWidth=3)
    elif d["kind"] == "company":
        net.add_node(n, label=d["label"], title=f"{d['title']} | degree {deg[n]}",
                     color="#4CAF50", size=35, borderWidth=2)
    else:
        net.add_node(n, label=d["label"], title=f"{d['title']} | degree {deg[n]}",
                     color="#64B5F6", size=15)
for u, v, d in G.edges(data=True):
    w = 1 + 7 * (math.log1p(d["total_value"]) - math.log1p(lo)) / span
    net.add_edge(u, v, width=w,
                 title=f"txns={d['n_txn']:,} value=${d['total_value']:,.0f}")


## Cell 6 — Save visualization

In [ ]:
import os

os.makedirs(os.path.dirname(HTML_OUT), exist_ok=True)
net.save_graph(HTML_OUT)
print(f"Saved: {HTML_OUT} ({os.path.getsize(HTML_OUT):,} bytes)")
